In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats
from tools import manhattan_plot, downsample_gwas
import gc
plt.rcParams['pdf.fonttype'] = 42

# Miami plot

In [ ]:
concat = []
concat_skew = []
concat_exome = []

for chrom in range(1, 22+1):
    print(f"reading chr{chrom}")
    df = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/common_var/restricted/chr{chrom}.common_var.fisher_p.GWAS.txt.gz', sep='\t')
    df['LOG10P'] = -np.log10(df['fisher_p'])
    df = df.query('carrier_mCA+carrier_control > 0.01*1e6 and carrier_mCA+carrier_control < 0.99*1e6')
    if chrom == 14: df = df.query('pos > 20e6') # filter out artifacts due to genotyping near acrocentric centromeres
    if chrom == 22: df = df.query('pos > 16e6')
    df = downsample_gwas(df)
    concat.append(df)

    df_skew = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/common_var/restricted/chr{chrom}.common_var.binom_p.GWAS.txt.gz', sep='\t')
    df_skew['LOG10P'] = -np.log10(df_skew['binom_p'])
    df_skew = downsample_gwas(df_skew)
    concat_skew.append(df_skew)

    df_exome = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/chr{chrom}.LoF.protein_coding.CN-LOH.GWAS.txt.gz', sep = '\t')
    df_exome['LOG10P'] = -np.log10(np.maximum(df_exome['fisher_p'], np.repeat(1e-300, len(df_exome))))
    df_exome = downsample_gwas(df_exome)
    concat_exome.append(df_exome)

df = pd.concat(concat)
df = df.rename({'pos': 'GENPOS'}, axis = 1)
df["CHROM"] = [int(x[3:]) for x in df['chr']]

df_exome = pd.concat(concat_exome)
df_exome = df_exome.rename({'pos': 'GENPOS'}, axis = 1)
df_exome["CHROM"] = [int(x[3:]) for x in df_exome['chr']]

df_skew = pd.concat(concat_skew)
df_skew = df_skew.rename({'pos': 'GENPOS'}, axis = 1)
df_skew["CHROM"] = [int(x[3:]) for x in df_skew['chr']]

gc.collect()


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 5), dpi=300)

manhattan_plot(df_exome.query('ref=="ref"'), ax[0], max_p=50)
ax[0].axhline(np.log10(1.2e5), c='k', linestyle='--')
ax[0].set_ylabel('-log10(p)')
ax[0].set_ylim(0, 51)

manhattan_plot(df, ax[1], max_p=50)
ax[1].axhline(np.log10(5e8), c='k', linestyle='--')
ax[1].set_ylabel('-log10(p)')
ax[1].invert_yaxis()
ax[1].set_ylim(51, 0)
ax[1].xaxis.tick_top()
ax[1].xaxis.set_ticklabels([])


ax[0].spines['right'].set_visible(False)
ax[0].spines['top'].set_visible(False)
ax[0].spines['bottom'].set_visible(False)
ax[1].spines['right'].set_visible(False)
ax[1].spines['top'].set_visible(False)
ax[1].spines['bottom'].set_visible(False)

ax[0].text(0.5, 1, 'Burden test: association of rare coding variants with CN-LOH in cis', ha='center', va='bottom', transform=ax[0].transAxes, fontsize=14)
ax[1].text(0.75, 0.67, 'DLK1', fontstyle='italic', transform=ax[1].transAxes)
ax[1].text(0.5, 0, 'Common variant associations with CN-LOH in cis', ha='center', va='top', transform=ax[1].transAxes, fontsize=14)
fig.tight_layout()
plt.savefig('rare_vs_common_miami.pdf', bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(8, 10), dpi=300)

manhattan_plot(df_exome.query('ref=="ref"'), ax[0], max_p=50)
manhattan_plot(df_exome.query('ref!="ref"'), ax[1], max_p=50)
manhattan_plot(df, ax[2], max_p=50)
manhattan_plot(df_skew.query('variant!="chr1:63735:IG"'), ax[3], max_p=50)


ax[0].text(0.5, 1, 'Burden Fisher\'s exact GWAS', ha='center', va='bottom', transform=ax[0].transAxes, fontsize=14)
ax[0].axhline(-np.log10(1.2e-5), c='k', linestyle='--')
ax[1].text(0.5, 1, 'Rare variant Fisher\'s exact GWAS', ha='center', va='top', transform=ax[1].transAxes, fontsize=14)
ax[1].axhline(-np.log10(5e-8), c='k', linestyle='--')
ax[2].text(0.5, 1, 'Common variant Fisher\'s exact GWAS', ha='center', va='bottom', transform=ax[2].transAxes, fontsize=14)
ax[2].axhline(-np.log10(5e-8), c='k', linestyle='--')
ax[3].text(0.5, 1, 'Common variant allelic shift GWAS', ha='center', va='top', transform=ax[3].transAxes, fontsize=14)
ax[3].axhline(-np.log10(5e-8), c='k', linestyle='--')
fig.tight_layout()
for axis, label in zip(ax, ['a', 'b', 'c', 'd']):
    axis.set_ylabel(r'$-\log_{10}(p)$')
    axis.set_ylim(0, 51)
    axis.spines['right'].set_visible(False)
    axis.spines['top'].set_visible(False)
    axis.text(-0.1, 1.05, label, transform=axis.transAxes, fontsize=16, va='bottom', ha='left')
plt.savefig('burden_rare_var_common_var_allelic_shift_manhattan.pdf', bbox_inches='tight')
plt.show()

# Polygenic drive heatmap

In [ ]:
blood_traits = [
    'blood_PLATELET_COUNT_v2', 
    'blood_RED_COUNT_v2', 
    'blood_BASOPHIL_COUNT_v2',
    'blood_NEUTROPHIL_COUNT_v2',
    'blood_EOSINOPHIL_COUNT_v2',
    'blood_MONOCYTE_COUNT_v2',
    'blood_LYMPHOCYTE_COUNT_v2',
    'blood_WHITE_COUNT_v2',
]

zscores = []
x_label = []
n = []
for chrom in range(1,22+1):
    df = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/polygenic_drive/results/chr{chrom}.differential_PRS.txt.gz', sep='\t')
    df = df[df['event_arm'] != f"chr{chrom}pq"]
    df = df.groupby(['event_arm', 'trait']) \
        .agg(
            mean_prs=('differential_PRS', np.mean),
            sem_prs=('differential_PRS', lambda x: np.std(x)/np.sqrt(len(x))),
            z_score=('differential_PRS', lambda x: np.mean(x)/(np.std(x)/np.sqrt(len(x)))),
            num_events=('differential_PRS', len)
        ) \
        .reset_index() \
        .pivot(index='trait', columns='event_arm', values=['z_score', 'num_events']) 
    df = df.loc[blood_traits]
    x_label.append(df['z_score'].columns.to_numpy()) 
    zscores.append(df['z_score'].to_numpy())
    n.append(df['num_events'].to_numpy())

zscores = np.hstack(zscores)
x_label = np.hstack(x_label)
n = np.hstack(n)

meta_analysis = ((zscores * np.sqrt(n)).sum(axis = 1)/ np.sqrt(n.sum(axis = 1)))[:, None]
zscores = np.hstack([zscores, meta_analysis])
x_label = np.array([x[3:] for x in x_label]+ ['Any'])
y_label = [x[6:-3].capitalize().replace('_', ' ') for x in blood_traits]

sorted_p = np.sort(scipy.stats.norm.sf(zscores).flatten())
bh = 0.05 * np.arange(1, len(sorted_p)+1) / len(sorted_p)
p_thresh = sorted_p[np.argmin(sorted_p < bh)-1]
fdr_sig = scipy.stats.norm.sf(zscores) <= p_thresh
bonferroni_sig = scipy.stats.norm.sf(zscores) <= 0.05 / len(sorted_p)


fig, ax = plt.subplots(figsize=(12, 2.2), dpi=300)
for i in range(zscores.shape[0]):
    for j in range(zscores.shape[1]):
        if bonferroni_sig[i, j]: ax.text(j, i, '**', ha='center', va='center', color='k')
        elif fdr_sig[i, j]: ax.text(j, i, '*', ha='center', va='center', color='k')
colorbar = ax.imshow(zscores, cmap='bwr', aspect='equal', vmin=-10, vmax=10)
ax.set_xticks(ticks=np.arange(len(x_label)), labels=x_label, rotation=90, fontsize=8)
ax.set_yticks(ticks=np.arange(len(y_label)), labels=y_label)
ax.axvline(len(x_label)-1.5, color='k', linewidth=1)
ax.xaxis.tick_top()
cbar = fig.colorbar(colorbar, ax=ax, aspect=10)
cbar.ax.set_title("z")
plt.tight_layout()
for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(1)
plt.savefig('polygenic_heatmap.pdf', bbox_inches='tight', transparent=True)
plt.show()

# Blood GWAS lookup

In [ ]:
xs = []
ys = []
lowers = []
uppers = []

concord = []
discord = []

df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/polygenic_drive/sumstats/blood_WHITE_COUNT_v2.AS.sumstats.txt.gz', sep='\t')
df['sign'] = ((df['carrier_overrep'] - df['carrier_underrep']) * df['beta']) > 0
df = df.query('MAF>0.001')

width = 0.01
for thresh in np.arange(0.01, 0.05+width, width):
    if thresh == 0.05: top = 1
    else: top = thresh+width
    keep_var = (np.abs(df['beta'])>thresh) & (df['carrier_overrep'] != df['carrier_underrep'])

    n = keep_var.sum()
    a = df[keep_var]['sign'].sum()
    b = n - a

    concord.append(a)
    discord.append(b)

    lower = scipy.stats.beta(1/2+a, 1/2+b).ppf(0.025)
    upper = scipy.stats.beta(1/2+a, 1/2+b).ppf(0.975)
    print(lower, upper)
    xs.append(thresh)
    ys.append(a/n)
    lowers.append(a/n-lower)
    uppers.append(upper-a/n)

xs = np.array(xs)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, dpi=150)

ax1.bar(xs-width*0.9/2, concord, width=width*0.9/2, align='edge', color='k', label='CN-LOH direction consistent\nwith effect on WBC')
ax1.bar(xs, discord, width=width*0.9/2, align='edge', color='lightgray', label='CN-LOH direction opposite\nof effect on WBC')
ax1.legend(frameon=False, fontsize=14)
ax1.set_ylabel('Number of variants', fontsize=14)



ax2.bar(xs, np.array(ys)*100, width=width * 0.9, yerr=(np.array(lowers)*100, np.array(uppers)*100), capsize=10, color='gray')
ax2.axhline(50, color='k')
ax2.set_ylim(50, 100)
ax2.set_ylabel('% variants with CN-LOH direction\n' + 'consistent with WBC B sign', fontsize=14)
ax2.set_xlabel('Effect size on white blood count (|B|, s.d. units)', fontsize=14)
ax2.set_xticks([0.01, 0.02, 0.03, 0.04, 0.05])
ax2.set_xticklabels(['>0.01', '>0.02', '>0.03', '>0.04', '>0.05'])

plt.tight_layout()
plt.savefig('blood_sumstats_skew_enrichment.pdf', bbox_inches='tight', transparent=True)
plt.show()

# Common var h2

In [ ]:
import glob
import os
h2s = {}
ses = {}
for fname in glob.glob('/mnt/project/lohdata/david/mCAs_WGS/polygenic_drive/h2/*hsq'):
    chrom_arm = os.path.basename(fname).split('.')[0]
    with open(fname) as f:
        for line in f:
            line=line.strip().split()
            if line[0] == 'V(G)/Vp_L': 
                h2 = float(line[1])
                se = float(line[2])
            if line[0] == 'n':
                n = int(line[1])
                if n > 200: 
                    h2s[chrom_arm] = h2
                    ses[chrom_arm] = se

In [ ]:
meta_numer = 0
meta_denom = 0

xlabels = []
xticks = []

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
for i, name in enumerate(sorted(h2s, key=h2s.get)):
    if name == 'chrXpq':
        continue
    meta_numer += h2s[name]/(ses[name]**2)
    meta_denom += 1/(ses[name]**2)
    ax.errorbar(i, h2s[name], yerr=ses[name]*1.96, fmt='o', color='goldenrod')
    xlabels.append(name[3:])
    xticks.append(i)
ax.axhline(0, color='k', linestyle='--')
ax.axvline(i, color='gray', linestyle='--')
ax.errorbar(i+1, meta_numer/meta_denom, np.sqrt(1/meta_denom)*(1.96), fmt='o', color='goldenrod')
xlabels += ['CN-LOH\naverage']
xticks += [i+1]
ax.set_xticks(xticks, xlabels, rotation=90)
for label in ax.get_xticklabels():
    label.set_verticalalignment('center')
ax.tick_params(axis='x', which='both', length=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.set_ylim(-0.9, 0.9)
ax.set_ylabel('Polygenic contribution\n' + r'to CN-LOH directionality (hg2)', fontsize=14)
ax.set_xlabel(r'Chromosome arm', fontsize=14)
plt.savefig('h2_common_var.pdf', bbox_inches='tight', transparent=True)
plt.show()

# h2 calibration

In [ ]:
import itertools
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/polygenic_drive/h2/h2_calibration_simulations.txt.gz', sep='\t')
df['coverage'] = (df['reml'] - 2*df['reml_se'] <= df['h2']) & (df['reml'] + 2*df['reml_se'] >= df['h2'])
df_plot = df.groupby(['h2', 'n', 'm']).agg(
    reml=('reml', 'mean'),
    std = ('reml', 'std'),
    num =('reml', 'count'),
    coverage=('coverage', 'mean'),
).query('num>50').reset_index()

shape_map = {
    1000: 'o',
    5000: 's',
    10000: '^',
}

color_map = {  
    100: 'orchid',
    500: 'darkseagreen',
    1000: 'lightcoral',
}

fig, ax = plt.subplots(2, 1, figsize=(10, 8), dpi=150)
h2s = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
for h2 in h2s:
    ax[0].hlines(y=h2, color='k', linestyle=':', xmin=h2*10, xmax=h2*10+0.2*9)
    for i,(n,m) in enumerate(itertools.product([100, 500, 1000], [1000, 5000, 10000])):
        group = df_plot.query('h2==@h2 and n==@n and m==@m')
        x = h2*10+i*0.2
        if len(group)==0:
            continue
        else:
            reml = group['reml'].item()
            se = (group['std']/np.sqrt(group['num'])).item()
        ax[0].errorbar(
            x, reml, 1.96*se, 
            fmt=shape_map[m], 
            color=color_map[n], 
            markersize=8, 
            capsize=2
        )
    
        # if(group['coverage'].item() == 1): continue
        covered = (group['coverage'] * group['num']).item()
        lower = scipy.stats.beta(1/2+group['num'].item()-covered, 1/2+covered).ppf(0.025)
        if(group['coverage'].item() == 1): lower=0
        upper = scipy.stats.beta(1/2+group['num'].item()-covered, 1/2+covered).ppf(0.975)
        ci = [[1-group['coverage'].item() - lower], [upper - 1 + group['coverage'].item()]]
        ax[1].bar(
            x, 1-group['coverage'].item(), 
            width = 0.2, 
            label=f'{h2:.2f}',
            color=color_map[n],
            capsize=2,
            edgecolor='k',
            alpha=0.5
        )
        ax[1].errorbar(
            x, 1-group['coverage'].item(), 
            yerr=ci, 
            fmt='none', 
            color=color_map[n],
            capsize=2,
        )
ax[1].axhline(y=0.05, color='k', linestyle=':')

ax[1].set_xticks(h2s*10+4*0.2, h2s)
ax[0].set_xticks(ax[1].get_xticks(), h2s)
ax[0].tick_params(axis='both', labelsize=12)
ax[1].tick_params(axis='both', labelsize=12)
ax[1].set_xlabel('Simulated h2', fontsize=14)
ax[1].set_ylabel('Frac. of 95% CIs\nnot covering simulated h2', fontsize=14)
ax[0].set_ylabel('Estimated heritability', fontsize=14)


handles = []
# Define simple shape legend
for m, shape in shape_map.items():
    handles.append(
        mlines.Line2D(
            [], [], 
            color='k', 
            marker=shape, 
            linestyle='None', 
            markersize=8, 
            label=f'#SNPs={m}'
        )
    )

for n, color in color_map.items():
    handles.append(
        mpatches.Patch(
            color=color,  
            label=f'#Ind.={n}'
        )
    )

# Create the legends
ax[0].legend(handles=handles, loc='lower right', ncol=2, frameon=False, fontsize=14)
ax[0].spines['top'].set_visible(False)
ax[0].spines['right'].set_visible(False)
ax[1].spines['top'].set_visible(False)
ax[1].spines['right'].set_visible(False)

for axis,label in zip(ax, ['a','b']):
    axis.text(-0.15, 1.1, label, transform=axis.transAxes, fontsize=24, va='top', ha='right')
plt.savefig('h2_calibration_simulations.pdf', bbox_inches='tight', transparent=True)
plt.show()
